# AI 트랜드

## 1. AI 에이전트
### 1-1. AI 에이전트란?
- AI 에이전트 : 사용자가 정한 목표 달성을 위해 AI 모델을 활용해 환경과 상호작용하는 시스템
- 추론 + 계획 + 도구를 사용한 실행이 하나로 맞물려 돌아감
- LLM -> 말로만 답함 vs. 에이전트 -> 목표만 주면 계획하고 직접 움직임

    <img src="https://lilianweng.github.io/posts/2023-06-23-agent/agent-overview.png" width="500" style="background-color:white; padding:10px;">

### 1-2. 에이전틱 AI의 작동방식(ReAct)
- 생성형 AI : 결과물 한 번 출력하고 끝
- 에이전틱 AI : 목표 달성할 때까지 추론과 행동 반복
- ReAct = Reasoning(추론) + Acting(행동)
    - 생각 -> 행동 -> 관찰 -> 생각 순환 반복
    - 관찰 결과에 따라 다음 생각이 달라짐 = 핵심

    <img src="https://lilianweng.github.io/posts/2023-06-23-agent/react.png" width="500" style="background-color:white; padding:10px;">

- ex. 배송 지연 CS 에이전트 : 주문 상태 확인 -> 지연 원인 확인 -> 환불안 작성 -> 담당자 승인 -> 환불 + 메시지 발송
- 폭주 방지를 위해 앱이 정해두는 것
    - 최대 반복 횟수, 비용 한도, 중복 실행 방지, 중요한 행동 전 사람 승인
    - 알아서 성공 X -> 정해진 범위 안에서 순환 반복 O
- 모델은 도구를 직접 실행 X -> 호출 요청만 내놓고 실제 실행은 앱이 함

### 1-3. 다중 에이전트
- MAS(Multi Agent System) : 여러 에이전트가 역할을 나눠 함께 일하는 구조
- 오케스트레이터 + 서브 에이전트
    - 오케스트레이터 : 일을 쪼갬 -> 누구에게 맡길지 결정 -> 결과 통합
    - 서브 에이전트 : 맡은 역할만 수행

    <img src="https://assets.claude.com/1198befc0b33726c45692ac40f764022f4de1bf2.png?w=3840&q=75&fm=webp&fit=max" width="500" style="background-color:white; padding:10px;">

- 역할을 나누는 이유
    - 단일 에이전트는 스스로 실수 잡기 어려움 -> 검토를 다른 에이전트에게
    - 서브 에이전트마다 자기 컨텍스트 창 -> 여러 갈래 동시 조사 가능
- 언제 쓰나
    - 고정 워크플로우 : 순서가 정해진 단순 작업
    - 단일 에이전트 : 도구 몇 개로 해결되는 작업
    - 다중 에이전트 : 따로 조사해서 합칠 수 있는 큰 작업
- 모든 역할에 비싼 모델 X -> 역할에 맞게 급이 다른 모델 섞어 씀


## 2. MCP(Model Context Protocol)
### 2-1. MCP이전의 도구 호출: 왜 표준이 필요한가
- Function Calling : 모델이 "이 함수를 이 인자로 불러 줘"라는 요청을 내놓는 기능
    1. 앱 -> 모델 : 질문 + 함수 목록
    2. 모델 : 부를 함수와 값 선택
    3. 앱 : 함수 실행
    4. 앱 -> 모델 : 결과 전달 -> 최종 답 작성
- n×m 문제
    - 표준 X : 앱마다 × 도구마다 연동 코드 따로 -> n×m개
    - 표준 O : 앱은 클라이언트 1번, 도구는 서버 1번 -> n+m개

    <img src="https://huggingface.co/datasets/mcp-course/images/resolve/main/unit1/1a.png" width="400" style="background-color:white; padding:10px;">

    <img src="https://huggingface.co/datasets/mcp-course/images/resolve/main/unit1/2.png" width="400" style="background-color:white; padding:10px;">

- Function Calling vs. MCP -> 경쟁 관계 X, 둘 다 씀
    - Function Calling : 모델이 도구 호출 요청을 내놓는 방법
    - MCP : 앱이 도구를 찾고, 연결하고, 호출하는 방법

### 2-2. MCP의 등장
- MCP : LLM 앱과 외부 데이터·도구가 통신하는 개방형 표준 프로토콜
    - 2024.11 Anthropic이 오픈소스로 공개
    - 2025.12 리눅스 재단에 기증 -> 업계 공동 표준
- 해결된 것
    1. n×m -> n+m
    2. 발견의 표준화 - 서버가 쓸 수 있는 도구 목록 반환
    3. 원격 서버 인증 통일 - OAuth
- 도구 정의
    - name : 도구 이름
    - description : 모델이 언제 이 도구를 쓸지 판단하는 근거 -> 한 줄로 모델 행동이 바뀔 수 있음
    - inputSchema : 입력 값의 형식 -> 엉뚱한 값이 못 들어오게 막는 역할

### 2-3. MCP의 구조
- Host : 사용자가 쓰는 AI 앱 -> 클라이언트 관리, 어떤 도구 호출을 허용할지 결정
- Client : 호스트 안의 통신 담당 -> 서버 1개당 1개
- Server : 도구·데이터를 MCP 규격으로 내놓는 프로그램 -> 실제 작업 수행

    <img src="https://huggingface.co/datasets/mcp-course/images/resolve/main/unit1/3.png" width="500" style="background-color:white; padding:10px;">

- 서버의 기본 요소
    - Tools : 실행할 수 있는 행동
    - Resources : 읽기 전용 데이터
    - Prompts : 재사용 가능한 템플릿
- 통신 : JSON-RPC 2.0 / 로컬 서버는 stdio, 원격 서버는 Streamable HTTP

### 2-4. MCP의 작동 흐름
- 질문 전 : 서버 연결(로그인, 권한 동의) -> 도구 목록 요청(tools/list) -> 도구 이름·설명·입력 형식 응답
- 질문 후
    1. 사용자 -> 호스트 : 질문
    2. 호스트 -> 모델 : 질문 + 도구 목록
    3. 모델 -> 호스트 : 도구 호출 요청만 (실행 X)
    4. 호스트 -> 클라이언트 -> 서버 : 허용 여부 확인 후 tools/call
    5. 서버 : 실제 작업 수행 후 결과 반환
    6. 모델 : 결과 보고 답변 생성
- 모델 = 도구 고르기 + 답 쓰기 / 연결·호출·실행 = 호스트·클라이언트·서버
- MCP 서버 뒤에서는 기존 업무 시스템(API, DB 등)을 호출 -> MCP가 업무 로직을 대신하는 건 X, 공통된 연결 규격일 뿐

### 2-5. MCP의 한계와 주의점
- 보안은 프로토콜이 아니라 호스트가 책임
    - MCP는 연결하는 법만 정함, 무엇을 허락할지는 X
- 위험 요소
    - 신뢰할 수 없는 서버 연결 -> 누구나 만들어 올릴 수 있음
    - 도구 오염(tool poisoning) -> description 안에 모델에게만 보이는 지시를 숨김
    - 너무 넓은 권한 -> 공개·비공개 저장소 모두 접근 가능하면 정보 유출 가능
- 프롬프트 주입(prompt injection) : 도구가 가져온 내용 안에 지시가 섞여 들어오는 것 -> 권한 넓을수록 피해 큼
- 안전하게 쓰려면
    - 공식·검증된 서버만 연결
    - 필요한 권한만 주기
    - 되돌리기 어려운 호출은 사람 승인
    - 처음 연결하는 서버는 도구 설명 확인


## 3. 데이터 구조화 - 그래프와 온톨로지
### 3-1. 왜 구조화가 필요한가
- 에이전틱 AI가 일하려면 구조화된 데이터 필요
- 흔한 방법은 RAG지만 문서를 벡터 DB로 만드는 것만으로는 한계
- Top-K 딜레마
    - Top-K : 질문과 유사도가 가장 높은 상위 k개 조각만 LLM에 전달하는 방식
    - k 작으면 -> 필요한 정보 놓침 / k 크면 -> 관련 없는 정보 섞임
    - 원인 : 청크를 하나씩 따로 채점함 -> 청크 간 연결을 모름
    - 정답이 연결을 따라가야 나오는 경우 유사도가 낮아서 못 찾음
- 사내 데이터는 ERP, CRM, 회의록 등에 흩어져 있음 -> 벡터 DB는 조각 간 관계를 모름 -> 맥락 없는 답변
- GIGO (Garbage In, Garbage Out) -> 구조화 안 된 데이터를 넣으면 구조화 안 된 답변
- 그래서 데이터 구조화 필요 -> 방법론이 그래프 구조

### 3-2. 그래프 구조와 팔란티어 온톨로지
- 그래프 : 데이터를 노드(사물) + 엣지(관계)로 표현하는 구조
    - 테이블 -> 무엇이 있는가 저장
    - 그래프 -> 어떻게 연결되어 있는가 저장
- 온톨로지 : 데이터를 개념과 관계로 구조화하는 방법론
- 팔란티어 온톨로지 : 그래프 구조를 기업 전체에 적용해 흩어진 사내 데이터를 하나의 관계망으로 묶은 것
- 구성요소
    - 객체 타입 / 객체 / 객체 세트 -> 테이블 스키마 / 행 / 여러 객체의 모음
    - 속성 / 속성값 -> 열 / 실제 값
    - 공유 속성 -> 여러 객체 타입에서 공통으로 쓰는 속성
    - 링크 타입 / 링크 -> 관계의 스키마 / 실제 관계
    - 액션 타입 -> 실제 변경과 행동까지 온톨로지 위에서 정의, 일반 DB와의 가장 큰 차이점
        - ReAct의 Act, MCP의 Tools와 같은 역할
    - 역할 (Roles) -> 객체 단위까지 접근 권한 제어, 에이전트도 사람처럼 권한을 받음

| 관계형 DB | 그래프 DB | 팔란티어 온톨로지 |
|:---:|:---:|:---:|
| 테이블 스키마 | - | 객체 타입 |
| 행 | 노드 | 객체 |
| 열 | - | 속성 |
| 조인 | 엣지 | 링크 타입 |
| - | - | 액션 타입 |

### 3-3. 실전에서는 어떻게 쓰이나
- 기업 스케일
    - 그래프 DB(Neo4j) -> 사기 탐지, 추천 엔진, 소셜 네트워크 분석 등에 활용
    - Uber -> 그래프 기반 에이전트 시스템으로 사내 챗봇 운영
- 개인/팀 스케일
    - LightRAG : GraphRAG의 장점은 살리고 비용을 줄인 가벼운 그래프 RAG
        - 검색 모드 : naive, local(엔티티 중심), global(테마 중심), hybrid
        - GraphRAG는 커뮤니티 요약을 미리 생성 vs. LightRAG는 생략 -> LLM 호출이 적어서 비용 낮음
        - Low-level 키 : 구체적인 엔티티 이름 -> 로컬 서치
        - High-level 키 : 추상적인 테마·개념 -> 글로벌 서치
    - graphify : 폴더 안의 코드, 문서 등을 그래프로 만들어주는 도구
    - claude-obsidian : 마크다운 지식 베이스를 쌓는 위키형 지식 그래프 -> 링크 타입이 없어서 엄밀한 온톨로지는 X
- 온톨로지 사례 : 의료 기기 제조사
    - 수백 개 시스템에 흩어진 데이터를 하나의 온톨로지로 통합
    - "납품이 왜 지연됐나?" -> 공장 -> 작업지시서 -> 물류 -> 고객 관계를 따라 한 번에 추론


## 4. XAI
### 4-1. 블랙박스가 왜 문제인가
- XAI(Explainable AI) : 설명 가능한 인공지능
- 딥러닝은 파라미터가 너무 많아서 내부 과정이 블랙박스
- ex. 피부암 진단 AI가 반창고 붙은 사진을 암으로 분류 -> 입력과 출력만 봐서는 발견 불가
- 문제인 이유
    - 규제 : 의료, 금융, 자율주행 분야에서 판단 근거 설명 요구
    - 디버깅 : 잘못 학습했을 때 어디가 문제인지 찾으려면 내부를 봐야 함

### 4-2. 기존 해결 시도들
- 회귀 분석 -> 완전한 화이트박스
    - 계수를 직접 읽을 수 있음
    - 한계 : 복잡한 패턴에서는 성능이 너무 낮음
- SHAP / LIME -> 사후적 설명
    - 모델은 그대로 두고 외부에서 각 입력 특성의 기여도를 근사 계산
    - SHAP : Shapley Value로 각 feature가 예측값에 기여한 정도 수치화
    - LIME : 샘플 하나 주변을 간단한 모델(선형 모델)로 근사해서 해석

    <img src="https://christophm.github.io/interpretable-ml-book/lime_files/figure-html/fig-lime-fitting-1.png" width="450" style="background-color:white; padding:10px;">

    - 한계 : 입력을 바꿔보며 역으로 추정한 것 -> 진짜 이유가 아닐 수 있음
- LLM + RAG -> 블랙박스로 블랙박스 설명
    - 모델의 입력, 출력을 GPT에게 주고 이유를 물어봄
    - 한계 : GPT도 블랙박스라 실제 내부 연산과 일치한다는 보장 X

| 방법 | 설명 가능성 | 성능 | 신뢰도 |
|:---:|:---:|:---:|:---:|
| 회귀 분석 | 완전 | 낮음 | 높음 |
| SHAP / LIME | 근사 | 유지 | 부분적 |
| LLM + RAG | 생성 | 유지 | 낮음 |

### 4-3. Mechanistic Interpretability(기계적 해석 가능성)
- 블랙박스 자체를 열어보자는 접근
- 무엇을 출력하는가 X -> 어떻게 그 답에 도달하는가 O
- 가중치, 뉴런, 어텐션 헤드를 직접 분석해서 기능이 어디에 어떻게 구현되어 있는지 회로 수준까지 역공학
- 일반 XAI vs. Mech. Int.
    - 일반 XAI : 외부에서 관찰, 입력-출력 관계, 사후적
    - Mech. Int. : 내부를 직접 해부, 가중치·회로 수준, 구조적
- Activation Patching : 모델 내부의 어느 위치가 답을 결정하는지 찾는 방법
    1. 답이 다른 입력 A, B 준비
    2. B 처리 중 특정 위치의 활성값을 A의 값으로 교체
    3. 출력이 A의 답으로 바뀌면 -> 그 위치가 핵심 경로

### 4-4. 핵심 개념 3가지
- Features (특징) : 모델이 입력에서 알아챈 개념
    - 개별 뉴런 또는 여러 뉴런의 조합(방향 벡터)에 저장
    - 개념이 등장하면 해당 뉴런의 활성값이 커짐
- Circuits (회로) : 여러 뉴런/어텐션 헤드가 연결되어 특정 기능을 수행하는 서브네트워크
    - 회로를 찾으면 기능 제거나 이식이 이론적으로 가능
- Superposition (중첩) : 하나의 뉴런이 여러 개념을 겹쳐서 저장하는 것
    - 개념 수 >> 뉴런 수 -> 방향을 조금씩 겹치게 배치
    - 겹쳐서 생기는 간섭은 특징이 드물게(sparse) 나타나서 감수함
    - 뉴런 하나 ≠ 개념 하나 -> 해석이 어려운 가장 큰 원인
    - 해결 도구 : Sparse Autoencoder(SAE) -> 중첩된 개념을 분리해서 추출

### 4-5. 왜 지금 중요한가
- AI 안전(AI Safety)과 연결 -> 모델이 원하는 대로 작동하는지 내부에서 확인 가능
    - 단순 검사 : 테스트 결과가 좋으니 괜찮다
    - Mech. Int. : 문제 되는 회로를 찾아 배포 전에 수정
- 모델이 커지고 위임하는 결정이 많아질수록 더 중요해짐
- Anthropic, DeepMind, Redwood Research 등이 집중 투자 중
- 아직 초기 단계지만 블랙박스를 외부에서 추정 -> 직접 여는 방향으로 바뀌는 중

# AI 트랜드

## 1. AI 에이전트
### 1-1. AI 에이전트란?
### 1-2. 에이전틱 AI의 작동방식(ReAct)
### 1-3. 다중 에이전트

## 2. MCP(Model Context Protocol)
### 2-1. MCP이전의 도구 호출: 왜 표준이 필요한가
### 2-2. MCP의 등장
### 2-3. MCP의 구조
### 2-4. MCP의 작동 흐름
### 2-5. MCP의 한계와 주의점

## 3. 데이터 구조화 - 그래프와 온톨로지
### 3-1. 왜 구조화가 필요한가
### 3-2. 그래프 구조와 팔란티어 온톨로지
### 3-3. 실전에서는 어떻게 쓰이나

## 4. XAI
### 4-1. 블랙박스가 왜 문제인가
### 4-2. 기존 해결 시도들
### 4-3. Mechanistic Interpretability(기계적 해석 가능성)
### 4-4. 핵심 개념 3가지
### 4-5. 왜 지금 중요한가


